In [ ]:
import numpy as np
from scipy.special import gamma
from scipy.stats import skew, kurtosis

In [ ]:
N, PATHS, SEED = 5_000, 15_000, 20260612
P_VALUES = (0.80, 0.85, 0.90, 0.95)
Q_VALUES = (0.25, 0.50, 0.75)


def summary(p, q, rng):
    a, b = 2 * p - 1, 2 * q - 1
    xi = np.empty((PATHS, N), dtype=np.int8)
    xi[:, 0] = np.where(rng.random(PATHS) < q, 1, -1)
    rows = np.arange(PATHS)
    for t in range(1, N):
        past = xi[rows, rng.integers(0, t, PATHS)]
        xi[:, t] = past * np.where(rng.random(PATHS) < p, 1, -1)
    Z = gamma(a + 1) * xi.sum(axis=1, dtype=np.int64) / N**a - b
    return Z.mean(), Z.var(), skew(Z), kurtosis(Z, fisher=False)

In [ ]:
streams = iter(np.random.SeedSequence(SEED).spawn(len(P_VALUES) * len(Q_VALUES)))
rows = [(p, q, *summary(p, q, np.random.default_rng(next(streams))))
        for p in P_VALUES for q in Q_VALUES]
hdr = f"{'p':>4} {'q':>4} {'mean':>9} {'var':>9} {'skew':>9} {'kurt':>9}"
print(hdr)
print("-" * len(hdr))
for p, q, m, v, s, k in rows:
    print(f"{p:>4.2f} {q:>4.2f} {m:>9.3f} {v:>9.3f} {s:>9.3f} {k:>9.3f}")
print()
for p, q, m, v, s, k in rows:
    print(f"  {p:.2f} & {q:.2f} & {m:.3f} & {v:.3f} & {s:.3f} & {k:.3f} \\\\")